# Aula 05 — como treinar uma cabeça sobre um modelo congelado

**Objetivo:** compreender o que é treinado, como os dados orientam a atualização dos pesos e como verificar se o resultado é útil. Ao final, você deverá conseguir explicar o percurso de uma entrada até sua classificação e distinguir treinamento, calibração e inferência.

Usaremos o Phi-4 como produtor de representações e treinaremos duas pequenas camadas de saída, chamadas **cabeças**. Os dados de um projeto existente tornam o processo executável; não é necessário aprender as regras desse projeto para compreender o método.

**Percurso principal:** execute as seções 1 a 10 em sequência para realizar o treinamento. A extração de representações dos 816 exemplos é a etapa demorada. **Demonstração sem esperar o treinamento:** a seção 11 tem uma única célula independente que usa a cabeça já treinada do GitHub. Você pode executar essa célula em uma sessão nova, sem executar as anteriores.


## 1. O que significa “treinar uma cabeça”?

Um modelo pré-treinado transforma texto em uma representação numérica. Nesta aula, mantemos esse modelo **congelado**: seus pesos não são atualizados. Treinamos uma camada que aprende a transformar a representação em uma previsão adequada ao nosso alvo.

```text
Texto → tokenização → Phi-4 CONGELADO → vetor com 5.120 valores
                                                  │
                         ┌────────────────────────┴────────────────────┐
                         ↓                                             ↓
               Cabeça de classificação                      Cabeça probabilística
                 3 escores → softmax                          1 escore → sigmoid
                         ↓                                             ↓
                   classe prevista                             p(A) e 1 − p(A)
```

| Conceito | Papel no processo | Neste experimento |
|---|---|---|
| Entrada `x` | Informação disponível para produzir a previsão. | Texto estruturado por um contrato. |
| Alvo `y` | Resposta usada como supervisão no treinamento. | Classe discreta ou frequência de um evento. |
| Representação `h` | Saída numérica do modelo base. | Último estado oculto do prompt completo, com 5.120 valores. |
| Cabeça `W, b` | Parâmetros que aprendemos. | Uma camada linear independente por tarefa. |
| Perda | Mede o desacordo entre previsão e alvo. | Entropia cruzada ou perda binária. |
| Otimizador | Atualiza os parâmetros a partir dos gradientes. | Adam. |

**Por que treinar apenas a cabeça?** Neste método, reutilizamos as representações do Phi-4 e reduzimos a quantidade de parâmetros atualizados. Isso torna o treino da camada final pequeno; não torna barata a execução do Phi-4 nem garante generalização. Não é fine-tuning de todos os pesos ou LoRA.

A cabeça de classificação tem `3 × 5.120 + 3 = 15.363` parâmetros; a probabilística tem `1 × 5.120 + 1 = 5.121`. São dois problemas de aprendizagem separados, com alvos e perdas diferentes.

**Antes de continuar:** qual componente precisa mudar quando a previsão está errada? Nesta aula, a atualização alcança `W` e `b`, não os pesos do Phi-4.


## 2. Preparar o ambiente

**O que faremos:** instalar as bibliotecas e definir onde serão salvos dados, cache de representações e cabeças. **Por quê:** a extração é demorada e deve poder ser retomada.

No Colab, usamos o PyTorch e o NumPy disponíveis. Instalamos apenas as versões específicas usadas pelo carregador. O corpo do modelo usa o dispositivo selecionado; o treino das cabeças ocorre na CPU. Para persistir entre sessões do Colab, monte seu Drive e altere `WORK_DIR` para uma pasta nele antes de iniciar. O corpo Phi-4 é baixado na revisão fixa do contrato. A janela é de 16.384 tokens e não há truncamento.

**Observe após executar:** a pasta de trabalho exibida. Ela deve ter espaço para os pesos base e os artefatos produzidos.


In [ ]:
!pip install -q transformers==5.0.0 safetensors==0.8.0 huggingface-hub==1.23.0


In [ ]:
from pathlib import Path
WORK_DIR = Path("treinamento-aula05")
WORK_DIR.mkdir(parents=True, exist_ok=True)


## 3. Obter entradas, alvos e código

**O que faremos:** clonar a `main` do GitHub e abrir o pacote de dados e scripts. **Por quê:** precisamos reproduzir o mesmo contrato e preservar o alinhamento entre cada entrada e seu alvo.

A pasta `treinamento/piloto-v01/pacote` contém scripts e dados. O checkout usa um único commit da `main`; os carregadores conferem os hashes e o contrato do pacote. O corpo Phi-4 é baixado separadamente.

| Arquivo | Conteúdo | Entra no prompt? |
|---|---|---|
| `dados/inputs.jsonl.gz` | Entradas das duas tarefas. | Sim, através do encoder de cada tarefa. |
| `dados/supervisao.jsonl` | Alvo, tarefa, partição e grupo de cada exemplo. | Não. |
| `protocolo.json` | Partições, hiperparâmetros e critérios de calibração. | Não. |
| `modelo.json` | Revisão, arquivos e contrato da representação. | Não; configura o carregamento. |

**Observe após executar:** são 816 exemplos alinhados. O mesmo índice liga entrada e supervisão para treinamento; ele não é uma informação fornecida ao modelo.


In [ ]:
!git clone -q --depth 1 --branch main https://github.com/AIConnection/missao-espacial.git missao-espacial

import sys, json
from pathlib import Path
REPO_DIR = Path("missao-espacial")
PACKAGE_DIR = REPO_DIR / "treinamento/piloto-v01/pacote"
sys.path.insert(0, str(PACKAGE_DIR / "ferramentas"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
from treinar import read_data
inputs, supervision = read_data(PACKAGE_DIR)
print("Exemplos:", len(inputs))


### 3.1. Separar entrada e supervisão

Vamos inspecionar **um exemplo real completo** da tarefa de classificação. Escolhemos uma entrada curta da partição de treino para facilitar a leitura; não mudamos seu conteúdo.

Nesta tarefa, a entrada contém `premise` (evidência disponível) e `hypothesis` (afirmação avaliada). O alvo é uma relação: suporte, insuficiente ou contradição. Essa semântica define **a tarefa de aprendizagem**, sem exigir que a aula ensine as regras do ambiente usado para gerar os textos.

O carregador acrescenta um prefixo constante à premissa e usa a instrução e o chat template originais. Eles fazem parte do contrato da representação e permanecem iguais no treino e na inferência. A instrução original menciona A/B/C, mas a cabeça treinada não gera letras: ela classifica a representação.

**Observe:** entrada e alvo são mostrados em blocos diferentes. O alvo e os metadados nunca são concatenados ao texto. Se entregássemos o gabarito na entrada, uma boa métrica poderia refletir vazamento em vez de aprendizagem.


In [ ]:
from treinar import LABELS
indice = min((r["index"] for r in supervision if r["task"] == "nli" and r["split"] == "train"),
             key=lambda i: len(json.dumps(inputs[i], ensure_ascii=False)))
print("ENTRADA COMPLETA — fornecida ao carregador:")
print(json.dumps(inputs[indice], ensure_ascii=False, indent=2))
print("\nALVO — usado na perda, fora do prompt:")
print({"classe": LABELS[supervision[indice]["target"]]})
print("\nMETADADOS — usados para organizar o experimento:")
print({k: supervision[indice][k] for k in ("task", "split", "group")})


### 3.2. Classes discretas e alvos probabilísticos

| Aspecto | Classificação | Estimativa de evento |
|---|---|---|
| Alvo de um exemplo | Uma classe entre três. | Frequência `k / n` entre 0 e 1. |
| Saída interna | Três probabilidades cuja soma é 1. | `p(A)`; a probabilidade de B é `1 − p(A)`. |
| Perda de treino | Entropia cruzada multiclasse. | Entropia cruzada binária com alvo fracionário. |
| Saída pública | Apenas a classe escolhida. | As duas probabilidades dos eventos contratados. |

Um alvo `28/32 = 0,875` significa que o evento A ocorreu em 28 das 32 continuações simuladas sob condições definidas. Não é uma certeza nem uma probabilidade universal. A cabeça aprende a aproximar essa frequência condicionada à entrada e ao procedimento de amostragem.

A identidade de A e B pertence ao contrato do projeto: A é entrega e B é retorno com sobrevivência. Para aprender o processo, o ponto importante é que são os dois resultados exclusivos representados e que **o alvo vem de contagens observadas**, não de uma opinião do modelo. A entrada desta tarefa contém contexto, objetivo, plano candidato, evento-alvo e condições de amostragem; estes textos condicionam a previsão, mas não constituem uma política de ações.

**Observe na célula seguinte:** distribuição de classes, distribuição das frequências e os dados de contagem de um exemplo. Uma massa de frequências perto de 1 pode tornar uma constante difícil de superar. Balancear classes na primeira tarefa não balanceia os eventos da segunda.


In [ ]:
from collections import Counter
print("CLASSIFICAÇÃO — quantidade por classe:")
print(dict(Counter(LABELS[r["target"]] for r in supervision if r["task"] == "nli")))
print("\nPROBABILIDADES — quantidade de exemplos por alvo k/n:")
print(dict(sorted(Counter(r["target"] for r in supervision if r["task"] == "forecast").items())))
exemplo_prob = next(r for r in supervision if r["task"] == "forecast" and 0 < r["target"] < 1)
print("\nUM ALVO REAL, SEPARADO DA ENTRADA:")
print(json.dumps({"target": exemplo_prob["target"], "counts": exemplo_prob["supervision"]["counts"],
                  "trials": exemplo_prob["supervision"]["trials"]}, ensure_ascii=False, indent=2))


## 4. Separar treino, calibração e teste

```text
Treino          → calcula normalização e atualiza W e b
Calibração      → escolhe temperatura com W e b já fixos
Teste reservado → mede resultado; não escolhe parâmetros
```

| Tarefa | Treino | Calibração | Teste | Total |
|---|---:|---:|---:|---:|
| Classificação | 420 | 90 | 90 | 600 |
| Frequência de evento | 144 | 36 | 36 | 216 |

Na classificação, cada partição é balanceada: 140 por classe no treino e 30 por classe em cada reserva. Exemplos relacionados são mantidos no mesmo grupo e um grupo não atravessa partições. Para a tarefa probabilística, as variantes de cada família estrutural ficam juntas.

**Por que separar por grupo?** Se uma versão de um exemplo entrar no treino e uma versão quase igual no teste, podemos medir reconhecimento de um padrão repetido. A separação por grupo reduz esse vazamento; não garante que o teste cubra todos os assuntos ou estratégias. As 6.912 continuações simuladas sustentam 216 alvos e não são 6.912 exemplos independentes de treino.

**O que faremos:** conferir os tamanhos das partições e os hiperparâmetros definidos antes da avaliação.


In [ ]:
protocol = json.loads((PACKAGE_DIR / "protocolo.json").read_text())
print("Exemplos por tarefa e partição:")
print(dict(Counter((r["task"], r["split"]) for r in supervision)))
print("Hiperparâmetros definidos antes do teste:", protocol["optimization"])


## 5. Extrair representações do Phi-4 congelado

**O que faremos:** primeiro baixar os arquivos do Phi-4; depois carregá-lo e executá-lo para cada entrada e salvar o último estado oculto do prompt completo. Cada entrada vira um vetor de 5.120 números; nenhuma supervisão é acrescentada ao prompt.

```text
Entrada + protocolo fixo → tokens → Phi-4 congelado → h ∈ R⁵¹²⁰ → cache
```

**Por que salvar as representações?** O corpo não muda entre épocas; podemos executar a etapa cara uma vez e reutilizar os vetores ao treinar as cabeças. O cache não é a cabeça e seus números não são probabilidades.

A extração também captura a projeção original A/B/C, usada como baseline da classificação. Essa projeção não é uma cabeça treinada em outro projeto.

**Observe:** ao concluir, o cache deve ter 816 linhas. A retomada verifica identidade do modelo, ambiente e hashes; não usa um cache parcial para treinar. Mudar dispositivo, runtime ou configuração exige outra pasta de trabalho. Essa etapa pode levar horas; para demonstrar o resultado sem esperar, vá à seção 11.

### 5.1. Baixar o corpo Phi-4

A clonagem do GitHub trouxe as cabeças treinadas, os dados e os scripts. **Os pesos base do Phi-4 vêm do Hugging Face**, do repositório e revisão registrados em `modelo.json`. A célula abaixo faz esse download explicitamente. São cerca de 8,2 GB; arquivos já em cache são reutilizados.

**Observe:** nome do modelo, revisão, progresso do download e pasta final. Ter arquivos no disco ainda não significa que o modelo foi carregado na memória ou executado.


In [ ]:
from huggingface_hub import snapshot_download
modelo_base = json.loads((PACKAGE_DIR / "modelo.json").read_text())
print("Phi-4 base:", modelo_base["repo"], "| Revisão:", modelo_base["revision"])
PHI4_DIR = snapshot_download(
    repo_id=modelo_base["repo"], revision=modelo_base["revision"],
    allow_patterns=[arquivo["path"] for arquivo in modelo_base["model_files"]])
print("Arquivos do Phi-4 no disco:", PHI4_DIR)


### 5.2. Carregar e executar o Phi-4 congelado

Agora `extract` recebe a pasta baixada, verifica os arquivos, carrega o Phi-4 na memória e o executa para cada entrada. Cada execução produz a representação que será salva no cache. Os pesos do Phi-4 ficam congelados.

```text
Arquivos do Phi-4 no disco → modelo carregado na memória → execução sobre tokens → vetor
```

O treinamento posterior das cabeças usa esses vetores: ele não precisa executar o Phi-4 novamente a cada época. A inferência de uma entrada nova precisa executar o corpo outra vez.


In [ ]:
from treinar import extract
extraction = extract(PACKAGE_DIR, WORK_DIR, model_directory=PHI4_DIR)
print("Representações concluídas:", len(extraction["rows"]))


## 6. Preparar os vetores e observar uma atualização

Antes de treinar, calculamos média e desvio **somente nos vetores de treino de cada tarefa**. Aplicamos a mesma transformação às reservas:

`z = (h − média_do_treino) / desvio_do_treino`

O desvio mínimo é 0,001 para evitar divisão por valores muito pequenos. Usar dados de teste para calcular a normalização incorporaria informação da reserva no ajuste.

A próxima célula demonstra **uma atualização da cabeça multiclasse**, usando as 420 representações reais de treino. É uma cabeça separada, inicializada com zeros, para observar a mecânica; ela não é exportada nem substitui o treino completo da seção 7.

```text
z → Wz + b (logits) → perda comparada a y
                           ↓
                     backward: gradientes
                           ↓
                     Adam: atualiza W e b
```

Entropia cruzada recebe logits; o PyTorch incorpora a transformação apropriada de modo numericamente estável. Não aplicamos softmax antes de `cross_entropy`. Com pesos e vieses zero, as três probabilidades começam em 1/3 e a perda começa em `−ln(1/3) ≈ 1,099`.

**Observe:** probabilidades, perda e norma do gradiente antes/depois do passo. Não suponha que cada passo reduza a perda: com a taxa 0,01 deste piloto, a primeira atualização pode aumentá-la. Isso não deve ser escondido ajustando o exemplo; compare com o histórico das 400 épocas na seção seguinte. Uma mudança nos pesos demonstra atualização, e uma perda menor no treino ainda não demonstra generalização. A L2 adicionada à perda penaliza a soma dos quadrados dos pesos, para limitar seu crescimento; o viés não recebe essa penalização neste método.


In [ ]:
import numpy as np, torch
from treinar import load_features, normalize
features, baseline_abc, cache_manifest = load_features(WORK_DIR)
indices_treino = [r["index"] for r in supervision if r["task"] == "nli" and r["split"] == "train"]
media, desvio = normalize(features[indices_treino])
z_treino = torch.tensor((features[indices_treino] - media) / desvio, dtype=torch.float32)
y_treino = torch.tensor([supervision[i]["target"] for i in indices_treino], dtype=torch.long)
print("Matriz de treino:", tuple(z_treino.shape), "| Alvos:", tuple(y_treino.shape))
cabeca_didatica = torch.nn.Linear(5120, 3)
torch.nn.init.zeros_(cabeca_didatica.weight)
torch.nn.init.zeros_(cabeca_didatica.bias)
otimizador = torch.optim.Adam(cabeca_didatica.parameters(), lr=0.01)
otimizador.zero_grad()
logits = cabeca_didatica(z_treino)
perda = torch.nn.functional.cross_entropy(logits, y_treino)
objetivo = perda + 0.1 * cabeca_didatica.weight.square().sum() / 2
print("Antes — perda:", perda.item(), "| p do primeiro exemplo:", logits[0].softmax(0).detach().tolist())
objetivo.backward()
print("Norma do gradiente dos pesos:", cabeca_didatica.weight.grad.norm().item())
otimizador.step()
with torch.no_grad():
    logits_depois = cabeca_didatica(z_treino)
    print("Depois — perda:", torch.nn.functional.cross_entropy(logits_depois, y_treino).item(),
          "| p do primeiro exemplo:", logits_depois[0].softmax(0).tolist())


## 7. Treinar as cabeças e calibrar suas probabilidades

A rotina abaixo treina **cabeças novas**, com todas as linhas das respectivas partições de treino. Cada uma recebe sua própria normalização.

| Escolha | Implementação | Racional e limite |
|---|---|---|
| Inicialização | Pesos e vieses zero. | Torna explícito que a cabeça começa sem preferir classes; não herda uma cabeça de outro domínio. |
| Otimizador | Adam, taxa de aprendizagem 0,01. | Usa gradientes para ajustar a camada. É uma escolha fixa deste piloto, sem busca de hiperparâmetros. |
| Duração | 400 épocas; batch completo de treino; semente 171. | Cada época atualiza com toda a partição. O teste não escolhe quando parar. |
| Regularização | `perda + 0,1 × ΣW² / 2`. | Penaliza pesos grandes. `train_loss` e objetivo regularizado são números distintos. |
| Classificação | Entropia cruzada para três classes. | Favorece atribuir probabilidade ao alvo discreto correto. |
| Evento | BCE com logits, alvo `k/n`. | Aproxima a frequência observada, preservando a incerteza do alvo. |

Depois do treino, `W` e `b` ficam fixos. **Calibração por temperatura:** dividimos os logits por `T` e escolhemos `T` pela menor perda na partição de calibração, na grade `[0,5; 0,75; 1; 1,5; 2; 3; 5; 10]`. A temperatura controla a concentração das probabilidades; não substitui aprender a relação correta. Uma temperatura positiva preserva o argmax da classificação.

A função `train` executa treinamento, calibração, avaliação das reservas e exportação. **Observe:** histórico de perda e temperatura escolhida por tarefa. O teste não entra em nenhuma dessas escolhas. Não prometemos que 400 épocas ou esses hiperparâmetros sejam ideais para qualquer dataset.


In [ ]:
from treinar import train
result = train(PACKAGE_DIR, WORK_DIR)
for task in ("nli", "forecast"):
    print("\nTAREFA:", task)
    print("Início do histórico:", result["tasks"][task]["training_history"][:2])
    print("Fim do histórico:", result["tasks"][task]["training_history"][-2:])
    print("Calibração:", result["tasks"][task]["calibration"])


## 8. Avaliar: o que os números permitem concluir?

### Classificação

| Métrica | O que mede | Por que usamos |
|---|---|---|
| Acurácia | Fração de classes previstas corretamente. | Resume acertos; pode ocultar uma classe difícil. |
| Precisão por classe | Entre as previsões dessa classe, quantas são corretas. | Mostra falsos positivos daquela classe. |
| Recall por classe | Entre os alvos dessa classe, quantos foram reconhecidos. | Mostra falsos negativos daquela classe. |
| Macro F1 | Média do F1 das três classes, com pesos iguais. | Cada classe influencia igualmente o resumo. |
| Matriz de confusão | Contagens entre alvo e previsão. | Localiza quais relações são confundidas. |
| NLL | Média de `−ln(p do alvo correto)`. | Penaliza especialmente erros confiantes; menor é melhor. |

Acurácia e F1 avaliam escolhas de classe. NLL também avalia a probabilidade atribuída ao alvo: `p=0,9` dá perda ≈ 0,105; `p=0,01` dá ≈ 4,605. A API pública pode retornar só uma classe, mesmo que a avaliação interna use probabilidades.

Compare com **uniforme** e com **Phi-4 A/B/C sem a cabeça treinada**. Um baseline informa se a aprendizagem acrescentou algo além de uma referência simples. Mostramos resultados crus e calibrados; não escondemos um resultado pior depois da calibração.

### Alvos probabilísticos

**Brier binário** é a média de `(p − y)²` para eventos observados; menor é melhor. Com `k` ocorrências em `n`, o cálculo correto sobre os eventos é `f(1−p)² + (1−f)p²`, em que `f=k/n`. Só comparar `(p−f)²` omite a variabilidade dos eventos. A **perda binária** é `−f ln(p) − (1−f) ln(1−p)` e penaliza probabilidades extremas incompatíveis com observações.

Compare a cabeça com uma **constante ajustada apenas no treino**. Se quase todos os eventos têm o mesmo resultado, essa constante pode funcionar bem. Bins de calibração comparam probabilidade média e frequência observada, mas poucos bins ocupados e poucas famílias limitam a conclusão. As continuações são agrupadas por contexto, não observações independentes universais.

**Perguntas para leitura:** a cabeça superou o baseline? A média esconde erros de uma classe? A calibração melhorou a perda? As partições e o tamanho da amostra sustentam a generalização que você está atribuindo ao resultado?


In [ ]:
from apoio_aula05 import mostrar_avaliacao
mostrar_avaliacao(result)
print("Relatório completo:", WORK_DIR / "resultado.json")


## 9. Usar as cabeças que acabamos de treinar

**Treinamento:** alvo → perda → gradiente → atualização. **Inferência:** entrada → representação → cabeça fixa → previsão; não há atualização nem uso do alvo para produzir a resposta.

O carregador verifica pesos, normalização, temperatura, código e revisão do modelo base. A mesma regra de preparação da entrada precisa ser usada: mudar o prompt, o modelo base ou a dimensão torna as representações incompatíveis com o contrato.

A célula demonstra uma entrada reservada por tarefa. Não gera ações: a classificação retorna um rótulo, e a outra cabeça retorna as duas probabilidades específicas do contrato. **Observe:** produzir uma resposta válida não prova que a resposta é correta; essa comparação pertence à avaliação.


In [ ]:
import gc
from inferir import PilotPredictor
print("Carregando o Phi-4 baixado e as cabeças treinadas nesta sessão...")
predictor = PilotPredictor(PACKAGE_DIR, WORK_DIR / "heads", model_directory=PHI4_DIR)
for task in ("nli", "forecast"):
    idx = next(r["index"] for r in supervision if r["task"] == task and r["split"] == "test")
    print("Entrada:", json.dumps(inputs[idx], ensure_ascii=False, indent=2))
    print("Executando Phi-4 → representação → cabeça fixa...")
    print("Saída:", predictor.predict(task, inputs[idx]))
del predictor
gc.collect()


## 10. Exportar e reproduzir

Uma cabeça utilizável precisa de mais que `W` e `b`: normalização, temperatura, ordem das classes, identidade do corpo e contrato de preparação das entradas também precisam acompanhar o artefato.

O ZIP final inclui pacote, cabeças e relatório; não inclui os 8,2 GB do corpo ou todo o cache de features. O corpo pode ser baixado pela revisão registrada. Para retomar extrações, preserve o cache em armazenamento persistente separado.

**Observe:** o arquivo exportado. Antes de usar em outro ambiente, confira o mesmo contrato; execução em um dispositivo não demonstra desempenho ou tempo em todos os dispositivos.


In [ ]:
from apoio_aula05 import exportar
export_path = exportar(PACKAGE_DIR, WORK_DIR)


## 11. Demonstração independente com a cabeça treinada do GitHub

**Esta célula pode ser executada diretamente em uma sessão nova.** Ela instala as dependências, baixa scripts, casos e cabeças da `main`, confere os hashes e carrega o Phi-4 base compatível. Não precisa das representações ou cabeças produzidas nas seções anteriores, não treina e não recalibra.

O objetivo é observar o processo de inferência e testar transferência para outros assuntos. Mudamos os textos; mantemos pesos e preparação da entrada. A cabeça de eventos não é avaliada fora de seu contrato específico.

```text
Entrada nova → MESMO encoder e Phi-4 → MESMA normalização e cabeça → classe
                                                                  │
                                                gabarito externo → comparação
```

A entrada contém apenas `premise` e `hypothesis`. A hipótese é uma afirmação a ser avaliada. O modelo recebe também o prefixo fixo e a instrução original, preservados pelo encoder. A função de apoio mostra o prefixo fixo; a célula mostra os dois campos de cada entrada. A classificação é suporte, insuficiente ou contradição; “não foi confirmado” não significa “foi negado”. Esse é o significado da tarefa, não uma regra de estratégia do ambiente.

**Como ler a demonstração:** leia premissa e hipótese, preveja a classe e compare com a saída. O gabarito e a justificativa aparecem depois da previsão e não entram no modelo. Esses exemplos concretizam a avaliação; não é necessário estudar as regras do jogo.

Por padrão são 30 pares, 10 por classe: 24 bases e seis variantes relacionadas. Para uma demonstração curta, substitua `INDICES = list(range(30))` por `INDICES = [15, 18, 16, 11, 13, 24]`. Esses seis casos contrastam hipóteses sob uma mesma evidência. A numeração interna começa em zero.

São inferências novas, sem copiar previsões históricas. O primeiro uso ainda pode baixar os 8,2 GB do corpo; não repetir o treino elimina as 816 extrações de treino, não o custo de executar o modelo para cada novo caso.

**Limite da avaliação:** temas e formulações mudam juntos; as variantes também combinam paráfrase e informação irrelevante. O teste exploratório não isola a causa de erros e não prova generalização universal. Os rótulos foram definidos antes da inferência. A acurácia considera todos os casos executados; o macro F1 dá igual peso às três classes.

A célula marca três etapas: **1. download dos pesos base do Phi-4**, **2. carregamento do corpo e da cabeça do GitHub**, **3. execução do Phi-4 e classificação da representação**. O download vem do Hugging Face; os pesos da cabeça vêm do checkout do GitHub. Reutilizar o cache evita baixar arquivos novamente, mas a entrada nova ainda precisa passar pelo Phi-4.


In [ ]:
!pip install -q transformers==5.0.0 safetensors==0.8.0 huggingface-hub==1.23.0
!git clone -q --depth 1 --branch main https://github.com/AIConnection/missao-espacial.git missao-espacial-demo

import sys, gc
sys.path.insert(0, "missao-espacial-demo/scripts")
from apoio_aula05 import baixar_phi4, carregar_demo, avaliar_demo
print("1. Baixar os pesos base do Phi-4, ou reutilizar o cache:")
phi4_dir = baixar_phi4("missao-espacial-demo")
print("2. Carregar o Phi-4 congelado e a cabeça treinada do GitHub:")
modelo, pares, gabaritos = carregar_demo("missao-espacial-demo", model_directory=phi4_dir)
INDICES = list(range(30))

resultados = []
for index in INDICES:
    entrada = pares[index]
    print(f"\nCASO {index + 1:02d}")
    print("PREMISSA:\n" + entrada["premise"])
    print("HIPÓTESE:", entrada["hypothesis"])
    print("3. Executar Phi-4 → representação → cabeça → classificação:")
    saida = modelo.predict("nli", entrada)
    print("SAÍDA:", saida)
    print("GABARITO:", gabaritos[index]["expected"])
    print("JUSTIFICATIVA:", gabaritos[index]["rationale"])
    resultados.append({"index": index, "expected": gabaritos[index]["expected"], "output": saida})
del modelo
gc.collect()
relatorio = avaliar_demo("missao-espacial-demo", resultados)


## 12. Verificar o que você aprendeu

Explique, usando o notebook:

1. O que permanece congelado e quais pesos recebem gradientes?
2. Por que o alvo precisa ficar fora da entrada do modelo?
3. Por que o cache de representações pode ser reutilizado entre épocas?
4. Por que normalização, temperatura e pesos usam partições diferentes?
5. Por que reduzir a perda de treino não basta para demonstrar generalização?
6. Por que uma constante é um baseline necessário quando um evento domina a massa?
7. O que precisa acompanhar a cabeça para reproduzir sua inferência?

**Atividade:** escolha um erro da avaliação, identifique entrada, alvo e previsão e proponha um experimento que mude apenas um fator. Não altere o gabarito para concordar com o modelo e não use o teste para escolher hiperparâmetros.

### Referências do método

- [Phi-4 — relatório técnico](https://arxiv.org/abs/2412.08905): modelo base.
- [Alain e Bengio — linear probes](https://arxiv.org/abs/1610.01644): examinar representações com camadas treináveis; não valida por si só nosso classificador textual.
- [Kingma e Ba — Adam](https://arxiv.org/abs/1412.6980): atualização dos parâmetros.
- [Guo et al. — calibração](https://proceedings.mlr.press/v70/guo17a.html): calibração por temperatura.
- [Gneiting e Raftery — regras de pontuação](https://sites.stat.washington.edu/raftery/Research/PDF/Gneiting2007jasa.pdf): avaliação de previsões probabilísticas.

A implementação específica deste piloto está nos scripts versionados do GitHub. Os resultados devem ser interpretados sob as partições, contratos e condições efetivamente executados.
